In [1]:
# Gamelog Visuals
!pip install plotly
import plotly.express as px
from plotly.subplots import make_subplots
import pandas as pd
from datetime import datetime

In [2]:
TEAM_COLUMN_NAME = "Team"
DATE_COLUMN_NAME = "Date"
RESULT_COLUMN_NAME = "Result"
OPPONENT_COLUMN_NAME = "Opp"

AWAY_COLUMN_NUMBER = 5

IS_WIN_COLUMN_NAME = "is_win"
INFO_STRING_COLUMN_NAME = "info_string"
COUNT_COLUMN_NAME = "count"

In [3]:
# Axis choices
POINTS_COLUMN_NAME = "PTS"
REBOUNDS_COLUMN_NAME = "TRB"
ASSISTS_COLUMN_NAME = "AST"
PLUS_MINUS_COLUMN_NAME = "+/-"

NUM_DAYS_REST_SINCE_PREVIOUS_GAME_COLUMN_NAME = "num_days_rest"

In [4]:
INFO_COLUMN_NAMES = [POINTS_COLUMN_NAME, REBOUNDS_COLUMN_NAME, ASSISTS_COLUMN_NAME, PLUS_MINUS_COLUMN_NAME, NUM_DAYS_REST_SINCE_PREVIOUS_GAME_COLUMN_NAME]

In [5]:
x_column_name = None
y_column_name = None

other_stat_column_names = None

In [6]:
split = False

In [7]:
PLAYER_NAME_KEY = "player_name"
LABEL_KEY = "label"
FILEPATH_KEY = "filepath"
ORIGINAL_DF_KEY = "original_df"
SINGLE_DF_KEY = "single_df"
W_DF_KEY = "w_df"
L_DF_KEY = "l_df"

In [8]:
data = [
  {
    PLAYER_NAME_KEY: "Steph Curry",
    LABEL_KEY: "2024-2025 Regular Season",
    FILEPATH_KEY: "path/to/steph.csv"
  },
  {
    PLAYER_NAME_KEY: "Lebron James",
    LABEL_KEY: "2024-2025 Regular Season",
    FILEPATH_KEY: "path/to/lebron.csv"
  },
  # To add data:
  # 1) Go to basketball-reference.com and navigate to player gamelog
  # 2) Click "Hide Inactive/Did Not Play"
  # 3) Click "Share & Export" -> "Get table as CSV"
  # 4) Manually remove any divider and summary rows before saving the csv
  # 5) Add to this list a hash specifying player name, label, filepath
  # 6) Repeat if you like
]

In [10]:
for d in data:
  df = pd.read_csv(d[FILEPATH_KEY])
  d.update({ORIGINAL_DF_KEY: df})

In [11]:
def add_columns_to_df(df):
  df[IS_WIN_COLUMN_NAME] = df[RESULT_COLUMN_NAME].apply(lambda result: 1 if result[0] == "W" else 0)
  df[NUM_DAYS_REST_SINCE_PREVIOUS_GAME_COLUMN_NAME] = df[DATE_COLUMN_NAME].apply(lambda date: datetime.strptime(date, "%Y-%m-%d")).diff().apply(lambda timedelta: timedelta.days)
  return df

In [12]:
def process_df(df, is_win=None):
  df = add_columns_to_df(df)
  if split:
    df = df[df[IS_WIN_COLUMN_NAME] == is_win]

  info_strings = {}
  for idx, row in df.iterrows():
    away = row.iloc[AWAY_COLUMN_NUMBER]
    coordinate = (row[x_column_name], row[y_column_name])
    if coordinate not in info_strings:
      info_strings[coordinate] = []
    info_strings[coordinate].append(f"<i>{row[DATE_COLUMN_NAME]} {away if isinstance(away,str) else '_'}{row[OPPONENT_COLUMN_NAME]}</i> {row[RESULT_COLUMN_NAME]}: {' | '.join([name+str(row[name]) for name in other_stat_column_names])}")

    df.loc[idx, COUNT_COLUMN_NAME] = len(info_strings[coordinate])
    df.loc[idx, INFO_STRING_COLUMN_NAME] = f"<b>{x_column_name}: {row[x_column_name]} | {y_column_name}: {row[y_column_name]}</b><br>{'<br>'.join(info_strings[coordinate])}"

  return df

In [13]:
def update_data(x, y, s=False):
  global x_column_name
  global y_column_name
  global other_stat_column_names
  global split

  x_column_name = x
  y_column_name = y

  other_stat_column_names = INFO_COLUMN_NAMES.copy()
  if x in other_stat_column_names:
    other_stat_column_names.remove(x)
  if y in other_stat_column_names:
    other_stat_column_names.remove(y)

  split = s

  for d in data:
    if split:
      d.update({
        W_DF_KEY: process_df(d[ORIGINAL_DF_KEY], 1),
        L_DF_KEY: process_df(d[ORIGINAL_DF_KEY], 0)
      })
    else:
      d.update({
        SINGLE_DF_KEY: process_df(d[ORIGINAL_DF_KEY])
      })

In [14]:
def plot_axis(fig, r, c, player_name, label, df, is_win=None):
  if split:
    fig.layout.annotations[(r-1)*2 + (c-1)].text = f"{player_name} | {df.head(1).iloc[0][TEAM_COLUMN_NAME]} {df.shape[0]}{'W' if is_win == 1 else 'L'} | {label}"
  else:
    fig.layout.annotations[r-1].text = f"{player_name} | {df.head(1).iloc[0][TEAM_COLUMN_NAME]} {df[df[IS_WIN_COLUMN_NAME] == 1].shape[0]}W {df[df[IS_WIN_COLUMN_NAME] == 0].shape[0]}L | {label}"

  fig.add_traces(px.scatter(data_frame=df, x=x_column_name, y=y_column_name, color=IS_WIN_COLUMN_NAME if x_column_name == DATE_COLUMN_NAME else COUNT_COLUMN_NAME, custom_data=INFO_STRING_COLUMN_NAME).data[0], rows=r, cols=c)

In [15]:
def plot_data():
  if split:
    fig = make_subplots(rows=len(data), cols=2, subplot_titles=tuple('placeholder' for i in range(len(data)*2)), vertical_spacing=1/(len(data)*4), horizontal_spacing=.1)
  else:
    fig = make_subplots(rows=len(data), cols=1, subplot_titles=tuple('placeholder' for i in range(len(data))), vertical_spacing=1/(len(data)*4), horizontal_spacing=.1)

  for i, d in enumerate(data):
    if split:
      plot_axis(fig, i+1, 1, d[PLAYER_NAME_KEY], d[LABEL_KEY], d[W_DF_KEY], 1)
      plot_axis(fig, i+1, 2, d[PLAYER_NAME_KEY], d[LABEL_KEY], d[L_DF_KEY], 0)
    else:
      plot_axis(fig, i+1, 1, d[PLAYER_NAME_KEY], d[LABEL_KEY], d[SINGLE_DF_KEY])

  is_timeseries = x_column_name == DATE_COLUMN_NAME
  fig.update_coloraxes(showscale=False if is_timeseries else True)
  fig.update_traces(hovertemplate="%{customdata[0]}", line_color='#008000', mode="lines+markers" if is_timeseries else "markers")
  fig.update_xaxes(title_text=x_column_name)
  fig.update_yaxes(title_text=y_column_name)
  fig.update_layout(title_text=f"{x_column_name} vs {y_column_name}", height=len(data) * 500, xaxis_title=x_column_name, yaxis_title=y_column_name, hoverlabel=dict(font_family="Courier New"))
  fig.show()

In [16]:
update_data(DATE_COLUMN_NAME, PLUS_MINUS_COLUMN_NAME)
plot_data()

In [17]:
update_data(DATE_COLUMN_NAME, NUM_DAYS_REST_SINCE_PREVIOUS_GAME_COLUMN_NAME)
plot_data()

In [18]:
update_data(NUM_DAYS_REST_SINCE_PREVIOUS_GAME_COLUMN_NAME, PLUS_MINUS_COLUMN_NAME)
plot_data()

In [19]:
update_data(NUM_DAYS_REST_SINCE_PREVIOUS_GAME_COLUMN_NAME, PLUS_MINUS_COLUMN_NAME, True)
plot_data()

Double double, anyone? Fries with that?

In [20]:
update_data(REBOUNDS_COLUMN_NAME, ASSISTS_COLUMN_NAME, True)
plot_data()

In [21]:
# SAMPLE CSV
'''
Rk,Gcar,Gtm,Date,Team,,Opp,Result,GS,MP,FG,FGA,FG%,3P,3PA,3P%,2P,2PA,2P%,eFG%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,GmSc,+/-
1,957,1,2024-10-23,GSW,@,POR,W 140-104,*,25:04,4,10,.400,3,7,.429,1,3,.333,.550,6,6,1.000,0,9,9,10,2,0,2,0,17,21.3,23
2,958,2,2024-10-25,GSW,@,UTA,W 127-86,*,27:25,7,20,.350,4,13,.308,3,7,.429,.450,2,2,1.000,0,3,3,4,2,0,3,3,20,10.3,22
3,959,3,2024-10-27,GSW,,LAC,L 104-112,*,26:42,6,11,.545,4,7,.571,2,4,.500,.727,2,2,1.000,0,4,4,6,2,1,6,1,18,14.4,2
4,960,7,2024-11-04,GSW,@,WAS,W 125-112,*,24:05,7,15,.467,4,9,.444,3,6,.500,.600,6,6,1.000,1,2,3,6,0,0,2,1,24,19.4,9
5,961,8,2024-11-06,GSW,@,BOS,W 118-112,*,34:23,8,17,.471,4,9,.444,4,8,.500,.588,7,7,1.000,0,7,7,9,4,1,3,2,27,27.6,7
6,962,9,2024-11-08,GSW,@,CLE,L 117-136,*,23:46,5,10,.500,1,4,.250,4,6,.667,.550,1,1,1.000,0,1,1,2,2,0,6,0,12,4.7,-25
7,963,10,2024-11-10,GSW,@,OKC,W 127-116,*,36:30,13,23,.565,7,13,.538,6,10,.600,.717,3,4,.750,1,4,5,7,1,1,3,2,36,29.4,21
8,964,11,2024-11-12,GSW,,DAL,W 120-117,*,34:52,14,27,.519,5,12,.417,9,15,.600,.611,4,5,.800,0,6,6,9,1,2,4,2,37,29.0,24
9,965,12,2024-11-15,GSW,,MEM,W 123-118,*,26:01,4,9,.444,3,7,.429,1,2,.500,.611,2,2,1.000,1,7,8,5,4,0,3,2,13,14.8,9
10,966,13,2024-11-18,GSW,@,LAC,L 99-102,*,32:23,10,21,.476,6,15,.400,4,6,.667,.619,0,0,,1,6,7,6,0,1,3,1,26,19.3,-5
11,967,14,2024-11-20,GSW,,ATL,W 120-97,*,30:12,7,10,.700,4,6,.667,3,4,.750,.900,5,5,1.000,0,4,4,8,2,1,5,0,23,23.3,27
12,968,15,2024-11-22,GSW,@,NOP,W 112-108,*,33:08,6,13,.462,4,7,.571,2,6,.333,.615,3,3,1.000,0,7,7,7,1,1,4,3,19,15.8,4
13,969,16,2024-11-23,GSW,@,SAS,L 94-104,*,32:00,5,16,.313,3,10,.300,2,6,.333,.406,1,1,1.000,2,5,7,5,0,0,3,1,14,7.8,4
14,970,17,2024-11-25,GSW,,BRK,L 120-128,*,29:28,8,17,.471,8,16,.500,0,1,.000,.706,4,4,1.000,0,4,4,7,1,0,3,0,28,23.4,-4
15,971,19,2024-11-30,GSW,@,PHO,L 105-113,*,32:24,8,21,.381,3,10,.300,5,11,.455,.452,4,5,.800,1,6,7,4,0,0,0,1,23,16.0,4
16,972,20,2024-12-03,GSW,@,DEN,L 115-119,*,34:18,8,23,.348,4,15,.267,4,8,.500,.435,4,4,1.000,1,6,7,11,0,0,5,0,24,16.3,5
17,973,22,2024-12-06,GSW,,MIN,L 90-107,*,32:08,6,17,.353,3,9,.333,3,8,.375,.441,8,8,1.000,1,1,2,4,0,0,3,0,23,14.3,-3
18,974,23,2024-12-08,GSW,,MIN,W 114-106,*,34:36,8,18,.444,5,11,.455,3,7,.429,.583,9,11,.818,0,4,4,8,1,0,3,0,30,24.6,14
19,975,24,2024-12-11,GSW,@,HOU,L 90-91,*,33:40,8,17,.471,3,9,.333,5,8,.625,.559,0,1,.000,1,2,3,5,2,1,1,0,19,16.4,4
20,976,25,2024-12-15,GSW,,DAL,L 133-143,*,35:26,9,19,.474,7,13,.538,2,6,.333,.658,1,1,1.000,0,5,5,10,1,0,2,1,26,23.4,-10
21,977,26,2024-12-19,GSW,@,MEM,L 93-144,*,24:22,0,7,.000,0,6,.000,0,1,.000,.000,2,2,1.000,0,3,3,1,1,1,2,0,2,-1.6,-41
22,978,27,2024-12-21,GSW,@,MIN,W 113-103,*,34:13,10,21,.476,7,16,.438,3,5,.600,.643,4,5,.800,0,3,3,10,1,0,1,1,31,27.4,20
23,979,28,2024-12-23,GSW,,IND,L 105-111,*,35:25,2,13,.154,2,9,.222,0,4,.000,.231,4,4,1.000,0,5,5,7,0,3,3,1,10,6.8,6
24,980,29,2024-12-25,GSW,,LAL,L 113-115,*,35:37,14,24,.583,8,15,.533,6,9,.667,.750,2,2,1.000,0,1,1,6,0,0,4,3,38,26.1,-1
25,981,31,2024-12-28,GSW,,PHO,W 109-105,*,35:03,9,22,.409,4,13,.308,5,9,.556,.500,0,0,,2,4,6,6,1,0,5,3,22,11.8,20
26,982,32,2024-12-30,GSW,,CLE,L 95-113,*,29:05,4,14,.286,3,11,.273,1,3,.333,.393,0,0,,0,2,2,3,1,1,1,1,11,5.8,-2
27,983,33,2025-01-02,GSW,,PHI,W 139-105,*,29:49,11,15,.733,8,8,1.000,3,7,.429,1.000,0,0,,1,5,6,10,1,0,2,1,30,31.7,32
28,984,35,2025-01-05,GSW,,SAC,L 99-129,*,29:42,8,12,.667,4,8,.500,4,4,1.000,.833,6,6,1.000,0,7,7,0,1,0,4,3,26,18.7,-17
29,985,36,2025-01-07,GSW,,MIA,L 98-114,*,33:24,11,22,.500,8,17,.471,3,5,.600,.682,1,1,1.000,0,7,7,0,1,2,4,2,31,19.7,-10
30,986,37,2025-01-09,GSW,@,DET,W 107-104,*,36:08,5,21,.238,2,14,.143,3,7,.429,.286,5,5,1.000,1,8,9,6,2,0,3,0,17,10.6,-14
31,987,39,2025-01-13,GSW,@,TOR,L 101-104,*,35:00,9,17,.529,4,10,.400,5,7,.714,.647,4,4,1.000,1,6,7,7,1,0,4,2,26,21.3,0
32,988,40,2025-01-15,GSW,@,MIN,W 116-115,*,37:20,10,21,.476,7,12,.583,3,9,.333,.643,4,4,1.000,1,0,1,8,1,0,1,1,31,26.2,-1
33,989,41,2025-01-18,GSW,,WAS,W 122-114,*,34:14,10,22,.455,4,14,.286,6,8,.750,.545,2,2,1.000,1,4,5,6,1,1,1,1,26,21.0,7
34,990,42,2025-01-20,GSW,,BOS,L 85-125,*,27:27,6,16,.375,4,12,.333,2,4,.500,.500,2,2,1.000,0,3,3,4,2,0,3,3,18,10.7,-15
35,991,43,2025-01-22,GSW,@,SAC,L 117-123,*,33:50,6,11,.545,1,4,.250,5,7,.714,.591,1,1,1.000,1,2,3,12,0,0,3,1,14,15.0,2
36,992,44,2025-01-23,GSW,,CHI,W 131-106,*,30:47,8,19,.421,5,12,.417,3,7,.429,.553,0,0,,1,3,4,7,0,0,3,0,21,14.4,4
37,993,45,2025-01-25,GSW,,LAL,L 108-118,*,32:09,4,17,.235,2,9,.222,2,8,.250,.294,3,4,.750,0,1,1,9,2,0,3,2,13,7.1,-17
38,994,47,2025-01-29,GSW,,OKC,W 116-109,*,33:17,6,15,.400,5,10,.500,1,5,.200,.567,4,4,1.000,0,1,1,4,1,0,0,2,21,16.2,-2
39,995,48,2025-01-31,GSW,,PHO,L 105-130,*,31:13,5,14,.357,1,6,.167,4,8,.500,.393,3,3,1.000,1,2,3,3,0,2,2,1,14,8.6,-18
40,996,49,2025-02-03,GSW,,ORL,W 104-99,*,34:18,7,21,.333,2,12,.167,5,9,.556,.381,8,8,1.000,0,1,1,5,1,0,3,1,24,13.5,8
41,997,50,2025-02-05,GSW,@,UTA,L 128-131,*,34:44,12,31,.387,6,18,.333,6,13,.462,.484,2,2,1.000,0,1,1,7,0,1,3,2,32,17.2,-21
42,998,51,2025-02-06,GSW,@,LAL,L 112-120,*,37:05,13,35,.371,6,20,.300,7,15,.467,.457,5,5,1.000,2,5,7,4,1,1,4,3,37,19.9,-3
43,999,52,2025-02-08,GSW,@,CHI,W 132-111,*,33:56,10,19,.526,8,16,.500,2,3,.667,.737,6,8,.750,2,2,4,6,0,1,4,1,34,26.4,23
44,1000,53,2025-02-10,GSW,@,MIL,W 125-111,*,33:59,12,24,.500,6,16,.375,6,8,.750,.625,8,9,.889,0,6,6,4,0,0,2,0,38,28.2,8
45,1001,54,2025-02-12,GSW,@,DAL,L 107-111,*,37:10,9,23,.391,4,13,.308,5,10,.500,.478,3,4,.750,1,4,5,8,2,0,4,3,25,16.4,4
46,1002,55,2025-02-13,GSW,@,HOU,W 105-98,*,35:17,7,17,.412,5,13,.385,2,4,.500,.559,8,9,.889,1,4,5,3,0,0,1,2,27,19.7,7
47,1003,56,2025-02-21,GSW,@,SAC,W 132-108,*,30:45,7,13,.538,4,9,.444,3,4,.750,.692,2,2,1.000,0,1,1,6,2,0,1,0,20,19.2,0
48,1004,57,2025-02-23,GSW,,DAL,W 126-102,*,28:34,12,20,.600,3,8,.375,9,12,.750,.675,3,3,1.000,0,4,4,7,1,0,2,3,30,24.7,14
49,1005,58,2025-02-25,GSW,,CHO,W 128-92,*,23:39,6,14,.429,2,9,.222,4,5,.800,.500,1,1,1.000,0,4,4,6,1,0,2,2,15,11.2,26
50,1006,59,2025-02-27,GSW,@,ORL,W 121-115,*,34:18,16,25,.640,12,19,.632,4,6,.667,.880,12,12,1.000,0,4,4,3,2,0,4,0,56,46.2,15
51,1007,60,2025-03-01,GSW,@,PHI,L 119-126,*,36:27,10,18,.556,5,12,.417,5,6,.833,.694,4,4,1.000,1,4,5,13,1,0,3,2,29,28.6,12
52,1008,61,2025-03-03,GSW,@,CHO,W 119-101,*,30:15,6,14,.429,3,9,.333,3,5,.600,.536,6,8,.750,1,2,3,10,1,1,1,3,21,20.6,13
53,1009,62,2025-03-04,GSW,@,NYK,W 114-102,*,33:06,10,21,.476,5,9,.556,5,12,.417,.595,3,3,1.000,0,7,7,9,2,0,2,0,28,25.7,23
54,1010,63,2025-03-06,GSW,@,BRK,W 121-119,*,35:36,12,20,.600,7,13,.538,5,7,.714,.775,9,9,1.000,2,2,4,4,0,0,5,3,40,29.4,-16
55,1011,64,2025-03-08,GSW,,DET,W 115-110,*,33:27,8,22,.364,4,15,.267,4,7,.571,.455,12,12,1.000,1,2,3,4,1,0,4,1,32,20.5,11
56,1012,65,2025-03-10,GSW,,POR,W 130-120,*,34:01,6,14,.429,5,11,.455,1,3,.333,.607,7,7,1.000,1,1,2,3,2,0,4,0,24,17.7,6
57,1013,66,2025-03-13,GSW,,SAC,W 130-104,*,30:16,4,9,.444,2,6,.333,2,3,.667,.556,1,1,1.000,0,2,2,5,1,1,2,4,11,8.5,15
58,1014,67,2025-03-15,GSW,,NYK,W 97-94,*,35:00,8,20,.400,4,13,.308,4,7,.571,.500,8,9,.889,0,7,7,5,0,1,3,2,28,19.3,-5
59,1015,68,2025-03-17,GSW,,DEN,L 105-114,*,35:44,6,21,.286,4,14,.286,2,7,.286,.381,4,4,1.000,0,4,4,7,3,0,7,2,20,9.0,-7
60,1016,70,2025-03-20,GSW,,TOR,W 117-114,*,25:01,6,8,.750,2,4,.500,4,4,1.000,.875,3,5,.600,0,2,2,1,0,1,3,0,17,12.0,-7
61,1017,73,2025-03-28,GSW,@,NOP,W 111-95,*,33:54,7,21,.333,5,16,.313,2,5,.400,.452,4,5,.800,0,4,4,6,3,0,1,2,23,17.3,10
62,1018,74,2025-03-30,GSW,@,SAS,W 148-106,*,25:48,4,10,.400,2,4,.500,2,6,.333,.500,3,3,1.000,0,3,3,6,1,1,0,2,13,13.6,24
63,1019,75,2025-04-01,GSW,@,MEM,W 134-125,*,36:31,16,31,.516,12,20,.600,4,11,.364,.710,8,8,1.000,1,9,10,8,5,1,2,2,52,48.6,17
64,1020,76,2025-04-03,GSW,@,LAL,W 123-116,*,34:16,10,21,.476,4,11,.364,6,10,.600,.571,13,14,.929,1,2,3,6,0,0,2,1,37,29.0,-1
65,1021,77,2025-04-04,GSW,,DEN,W 118-104,*,32:11,13,24,.542,7,15,.467,6,9,.667,.688,3,3,1.000,1,1,2,5,2,0,2,2,36,28.1,6
66,1022,78,2025-04-06,GSW,,HOU,L 96-106,*,32:40,1,10,.100,1,8,.125,0,2,.000,.150,0,0,,0,2,2,8,0,0,4,0,3,-1.4,-4
67,1023,79,2025-04-08,GSW,@,PHO,W 133-95,*,25:55,9,17,.529,3,9,.333,6,8,.750,.618,4,4,1.000,1,8,9,6,1,1,2,0,25,23.7,31
68,1024,80,2025-04-09,GSW,,SAS,L 111-114,*,36:35,12,24,.500,5,14,.357,7,10,.700,.604,1,1,1.000,2,6,8,3,2,0,1,0,30,24.3,13
69,1025,81,2025-04-11,GSW,@,POR,W 103-86,*,27:20,6,14,.429,2,8,.250,4,6,.667,.500,0,0,,1,4,5,5,0,1,1,2,14,10.9,8
70,1026,82,2025-04-13,GSW,,LAC,L 119-124 (OT),*,38:00,10,20,.500,7,12,.583,3,8,.375,.675,9,9,1.000,1,2,3,6,2,0,8,2,36,24.7,-16
'''

'\nRk,Gcar,Gtm,Date,Team,,Opp,Result,GS,MP,FG,FGA,FG%,3P,3PA,3P%,2P,2PA,2P%,eFG%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,GmSc,+/-\n1,957,1,2024-10-23,GSW,@,POR,W 140-104,*,25:04,4,10,.400,3,7,.429,1,3,.333,.550,6,6,1.000,0,9,9,10,2,0,2,0,17,21.3,23\n2,958,2,2024-10-25,GSW,@,UTA,W 127-86,*,27:25,7,20,.350,4,13,.308,3,7,.429,.450,2,2,1.000,0,3,3,4,2,0,3,3,20,10.3,22\n3,959,3,2024-10-27,GSW,,LAC,L 104-112,*,26:42,6,11,.545,4,7,.571,2,4,.500,.727,2,2,1.000,0,4,4,6,2,1,6,1,18,14.4,2\n4,960,7,2024-11-04,GSW,@,WAS,W 125-112,*,24:05,7,15,.467,4,9,.444,3,6,.500,.600,6,6,1.000,1,2,3,6,0,0,2,1,24,19.4,9\n5,961,8,2024-11-06,GSW,@,BOS,W 118-112,*,34:23,8,17,.471,4,9,.444,4,8,.500,.588,7,7,1.000,0,7,7,9,4,1,3,2,27,27.6,7\n6,962,9,2024-11-08,GSW,@,CLE,L 117-136,*,23:46,5,10,.500,1,4,.250,4,6,.667,.550,1,1,1.000,0,1,1,2,2,0,6,0,12,4.7,-25\n7,963,10,2024-11-10,GSW,@,OKC,W 127-116,*,36:30,13,23,.565,7,13,.538,6,10,.600,.717,3,4,.750,1,4,5,7,1,1,3,2,36,29.4,21\n8,964,11,2024-11-12,GSW,,DAL,W 120-11